# Argus Vision — Build the Hard Subset $D_{hard}$

**Adversarial multi-agent visual debate for uncertainty-aware medical image classification.**

This notebook discovers the *hard* dermoscopic cases on which the two trained agents — **Agent A** (EfficientNet-B4) and **Agent B** (ViT-B/16) — disagree or are individually uncertain. These are exactly the cases where the Argus debate machinery is expected to add value.

Pipeline:
1. Load both trained agents from `../backend/checkpoints`.
2. Run inference over the **full test set**.
3. For every image compute the (squared) Jensen-Shannon divergence between the two agents and each agent's Shannon entropy.
4. Flag images where the debate trigger fires ($\tau_{JS}=0.25$, $\tau_{H}=0.8$).
5. Persist the hard-subset CSV (`image_path,true_label,pred_a,pred_b,js_divergence,entropy_a,entropy_b`).
6. Compare the class distribution of $D_{hard}$ against the full test set.
7. Visualize the 20 hardest cases (highest JS divergence).

The trigger maths is intentionally identical to `backend/ml/debate/trigger.py` so the offline subset matches the online behaviour exactly.

## 1. Environment, constants and the shared contract

In [ ]:
import os
import subprocess
from pathlib import Path

# Install the exact pinned training stack (torch, timm, scipy, pandas, ...).
subprocess.run(
    ["pip", "install", "-q", "-r", "requirements_training.txt"],
    check=True,
)
subprocess.run(["pip", "install", "-q", "scipy==1.13.0"], check=True)

from typing import Dict, List, Tuple  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402

# --- Shared contract: ISIC-8 taxonomy (exact index 0..7 order) ----------
CLASS_NAMES: List[str] = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
FULL_NAMES: Dict[str, str] = {
    "MEL": "Melanoma",
    "NV": "Melanocytic Nevus",
    "BCC": "Basal Cell Carcinoma",
    "AK": "Actinic Keratosis",
    "BKL": "Benign Keratosis",
    "DF": "Dermatofibroma",
    "VASC": "Vascular Lesion",
    "SCC": "Squamous Cell Carcinoma",
}
RISK_LEVELS: Dict[str, str] = {
    "MEL": "high", "NV": "low", "BCC": "high", "AK": "medium",
    "BKL": "low", "DF": "low", "VASC": "medium", "SCC": "high",
}
NUM_CLASSES: int = 8
IMAGE_SIZE: int = 224
IMAGENET_MEAN: List[float] = [0.485, 0.456, 0.406]
IMAGENET_STD: List[float] = [0.229, 0.224, 0.225]

# --- Debate-trigger thresholds (identical to the backend settings) ------
TAU_JS: float = 0.25
TAU_ENTROPY: float = 0.8

# --- Backbone identifiers (MUST match backend/ml/agents/*) --------------
AGENT_A_MODEL_NAME: str = "efficientnet_b4"
AGENT_B_MODEL_NAME: str = "vit_base_patch16_224.augreg_in21k_ft_in1k"

# --- Filesystem layout --------------------------------------------------
CHECKPOINT_DIR = Path("../backend/checkpoints")
AGENT_A_CHECKPOINT = CHECKPOINT_DIR / "agent_a_best.pth"
AGENT_B_CHECKPOINT = CHECKPOINT_DIR / "agent_b_best.pth"
ARTIFACT_DIR = Path("./artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
HARD_SUBSET_CSV = ARTIFACT_DIR / "hard_subset.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")
print(f"Checkpoint A exists: {AGENT_A_CHECKPOINT.exists()}")
print(f"Checkpoint B exists: {AGENT_B_CHECKPOINT.exists()}")

## 2. Load the test dataframe

We reuse the same ISIC 2019 ground-truth loader that trained the agents. The dataframe carries an absolute `image_path` and an integer `label` (0..7, in the canonical ISIC-8 order). A fixed-seed stratified split reproduces the held-out **test** partition exactly so that this subset is built on data the agents never trained on.

In [ ]:
from sklearn.model_selection import train_test_split

try:
    # The training notebooks ship a `dataset.py` helper and a `config.py`
    # with an AgentAConfig dataclass describing where the ISIC CSV lives.
    from config import AgentAConfig  # type: ignore
    from dataset import load_isic_dataframe  # type: ignore

    CFG = AgentAConfig()
    full_df = load_isic_dataframe(CFG)
    print(f"Loaded {len(full_df):,} labelled images via dataset.py.")
except Exception as exc:  # noqa: BLE001 - fall back to a manual scan.
    print(f"dataset.py loader unavailable ({exc}); scanning DATA_DIR.")
    DATA_DIR = Path(os.environ.get("ISIC_DATA_DIR", "./data"))
    gt_candidates = sorted(DATA_DIR.rglob("*GroundTruth*.csv"))
    if not gt_candidates:
        raise FileNotFoundError(
            "No ISIC ground-truth CSV found. Set ISIC_DATA_DIR or provide "
            "a dataset.py with load_isic_dataframe(CFG)."
        )
    raw = pd.read_csv(gt_candidates[0])
    image_col = raw.columns[0]
    onehot = raw[CLASS_NAMES].to_numpy()
    image_root = gt_candidates[0].parent
    records: List[Dict[str, object]] = []
    for i in range(len(raw)):
        name = str(raw.iloc[i][image_col])
        matches = list(image_root.rglob(f"{name}.jpg"))
        if not matches:
            continue
        records.append(
            {"image_path": str(matches[0]), "label": int(onehot[i].argmax())}
        )
    full_df = pd.DataFrame.from_records(records)
    print(f"Scanned {len(full_df):,} labelled images from {image_root}.")

full_df = full_df.reset_index(drop=True)
assert {"image_path", "label"}.issubset(full_df.columns), full_df.columns

# Reproduce the held-out test split (80/10/10 train/val/test by seed).
train_df, holdout_df = train_test_split(
    full_df, test_size=0.20, stratify=full_df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    holdout_df, test_size=0.50, stratify=holdout_df["label"], random_state=SEED
)
test_df = test_df.reset_index(drop=True)
print(f"Test-set size: {len(test_df):,} images.")
test_df.head()

## 3. Preprocessing transform

Both agents share the contract preprocessing: resize to 224×224, convert to a tensor, and normalize with the ImageNet statistics. Using a single deterministic transform guarantees the saliency and probability outputs are comparable across the two backbones.

In [ ]:
from PIL import Image
import torchvision.transforms as T

eval_transform = T.Compose(
    [
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


def load_tensor(image_path: str) -> torch.Tensor:
    """Load an image and return a preprocessed ``(1, 3, 224, 224)`` tensor.

    Args:
        image_path: Absolute path to a dermoscopic image on disk.

    Returns:
        A normalized float tensor with a leading batch dimension.
    """
    with Image.open(image_path) as handle:
        image = handle.convert("RGB")
    return eval_transform(image).unsqueeze(0)

## 4. Load the two trained agents

We construct the same `timm` backbones the backend serves and load the fine-tuned state dicts. If a checkpoint is missing we fall back to an ImageNet-pretrained backbone (mirroring `PRETRAINED_FALLBACK=True`) so the notebook still runs end-to-end — predictions are simply not clinically meaningful in that degraded mode.

In [ ]:
import timm
import torch.nn as nn


def build_agent(model_name: str, checkpoint: Path) -> nn.Module:
    """Construct a timm backbone and load its fine-tuned checkpoint.

    Args:
        model_name: The timm model identifier for the backbone.
        checkpoint: Path to the agent's ``.pth`` state dict.

    Returns:
        The model in eval mode on ``DEVICE``. Loads the checkpoint when
        present, otherwise falls back to ImageNet pretraining.
    """
    has_ckpt = checkpoint.exists()
    model = timm.create_model(
        model_name, pretrained=not has_ckpt, num_classes=NUM_CLASSES
    )
    if has_ckpt:
        state = torch.load(checkpoint, map_location=DEVICE)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        missing, unexpected = model.load_state_dict(state, strict=False)
        if missing:
            print(f"  [{model_name}] missing keys: {len(missing)}")
        if unexpected:
            print(f"  [{model_name}] unexpected keys: {len(unexpected)}")
        print(f"Loaded checkpoint {checkpoint.name}.")
    else:
        print(
            f"WARNING: {checkpoint.name} missing; using ImageNet fallback "
            f"for {model_name} (predictions not clinically meaningful)."
        )
    return model.eval().to(DEVICE)


agent_a = build_agent(AGENT_A_MODEL_NAME, AGENT_A_CHECKPOINT)
agent_b = build_agent(AGENT_B_MODEL_NAME, AGENT_B_CHECKPOINT)
print("Both agents ready.")

## 5. Trigger maths — mirror of `backend/ml/debate/trigger.py`

* `js_divergence` is the **squared** Jensen-Shannon distance (`scipy.spatial.distance.jensenshannon(..., base=2) ** 2`), giving a value in $[0, 1]$.
* `shannon_entropy` is the base-2 entropy in bits.
* The trigger fires when `js_divergence > tau_js` **or** `max(entropy_a, entropy_b) > tau_entropy`.

These are byte-for-byte the same formulas the live backend uses, so a case flagged here is a case the deployed system would debate.

In [ ]:
from scipy.spatial.distance import jensenshannon

_EPS = 1e-12


def shannon_entropy(probs: np.ndarray) -> float:
    """Compute the base-2 Shannon entropy (bits) of a probability vector.

    Args:
        probs: A non-negative probability vector of shape ``(NUM_CLASSES,)``.

    Returns:
        The Shannon entropy in bits as a Python float.
    """
    p = np.clip(probs, 0.0, None)
    return -float(np.sum(p * np.log2(p + _EPS)))


def js_divergence(probs_a: np.ndarray, probs_b: np.ndarray) -> float:
    """Compute the squared Jensen-Shannon divergence between two vectors.

    Args:
        probs_a: Agent A's probability vector ``(NUM_CLASSES,)``.
        probs_b: Agent B's probability vector ``(NUM_CLASSES,)``.

    Returns:
        The JS divergence in ``[0, 1]`` (0.0 when degenerate/non-finite).
    """
    distance = jensenshannon(probs_a, probs_b, base=2)
    divergence = float(distance) ** 2
    return divergence if np.isfinite(divergence) else 0.0


def trigger_fires(div: float, ent_a: float, ent_b: float) -> bool:
    """Replicate the backend debate-trigger decision.

    Args:
        div: The squared JS divergence between the two agents.
        ent_a: Agent A's Shannon entropy in bits.
        ent_b: Agent B's Shannon entropy in bits.

    Returns:
        ``True`` when the debate trigger fires under the contract thresholds.
    """
    return (div > TAU_JS) or (max(ent_a, ent_b) > TAU_ENTROPY)

## 6. Inference over the full test set

We run both agents over every test image, capture their softmax distributions, then compute the divergence/entropy metrics and the trigger decision per image.

In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm


@torch.no_grad()
def predict_probs(model: nn.Module, tensor: torch.Tensor) -> np.ndarray:
    """Return the softmax probability vector for a single image tensor.

    Args:
        model: A classifier in eval mode on ``DEVICE``.
        tensor: A ``(1, 3, 224, 224)`` preprocessed image tensor.

    Returns:
        A ``float64`` probability vector of shape ``(NUM_CLASSES,)``.
    """
    logits = model(tensor.to(DEVICE))
    probs = F.softmax(logits, dim=1)[0]
    return probs.detach().cpu().numpy().astype(np.float64)


rows: List[Dict[str, object]] = []
for record in tqdm(test_df.itertuples(index=False), total=len(test_df)):
    image_path = str(record.image_path)
    true_idx = int(record.label)
    try:
        tensor = load_tensor(image_path)
    except Exception as exc:  # noqa: BLE001 - skip unreadable images.
        print(f"Skipping unreadable image {image_path}: {exc}")
        continue

    pa = predict_probs(agent_a, tensor)
    pb = predict_probs(agent_b, tensor)

    div = js_divergence(pa, pb)
    ent_a = shannon_entropy(pa)
    ent_b = shannon_entropy(pb)

    rows.append(
        {
            "image_path": image_path,
            "true_label": CLASS_NAMES[true_idx],
            "pred_a": CLASS_NAMES[int(pa.argmax())],
            "pred_b": CLASS_NAMES[int(pb.argmax())],
            "js_divergence": div,
            "entropy_a": ent_a,
            "entropy_b": ent_b,
            "fired": trigger_fires(div, ent_a, ent_b),
        }
    )

results_df = pd.DataFrame(rows)
print(f"Scored {len(results_df):,} test images.")
results_df.head()

## 7. Flag and persist the hard subset $D_{hard}$

Every image whose trigger fires becomes part of $D_{hard}$. We save it with exactly the contract columns: `image_path,true_label,pred_a,pred_b,js_divergence,entropy_a,entropy_b`.

In [ ]:
HARD_COLUMNS = [
    "image_path",
    "true_label",
    "pred_a",
    "pred_b",
    "js_divergence",
    "entropy_a",
    "entropy_b",
]

hard_df = results_df[results_df["fired"]].copy()
hard_df = hard_df.sort_values("js_divergence", ascending=False)
hard_df = hard_df.reset_index(drop=True)
hard_df[HARD_COLUMNS].to_csv(HARD_SUBSET_CSV, index=False)

fired_count = int(results_df["fired"].sum())
fired_frac = fired_count / max(len(results_df), 1)
disagree = int((results_df["pred_a"] != results_df["pred_b"]).sum())
print(f"Trigger fired on {fired_count:,} / {len(results_df):,} "
      f"({fired_frac:.1%}) of test images.")
print(f"Raw agent disagreement (argmax differs): {disagree:,} images.")
print(f"Saved hard subset -> {HARD_SUBSET_CSV.resolve()}")
hard_df[HARD_COLUMNS].head(10)

## 8. Class distribution: $D_{hard}$ vs full test set

If the hard subset over-represents the high-risk malignant classes (MEL, BCC, SCC) relative to the full test set, that confirms the trigger is concentrating effort on the clinically consequential cases.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


def class_fractions(frame: pd.DataFrame) -> np.ndarray:
    """Return the per-class fraction vector in canonical ISIC-8 order.

    Args:
        frame: A dataframe with a ``true_label`` column of ISIC class codes.

    Returns:
        A length-8 ``float64`` array of class frequencies summing to 1.0
        (all zeros when the frame is empty).
    """
    counts = frame["true_label"].value_counts()
    vec = np.array([float(counts.get(name, 0)) for name in CLASS_NAMES])
    total = vec.sum()
    return vec / total if total > 0 else vec


full_frac = class_fractions(results_df)
hard_frac = class_fractions(hard_df)

x = np.arange(NUM_CLASSES)
width = 0.4
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width / 2, full_frac, width, label="Full test set", color="#6B7280")
ax.bar(x + width / 2, hard_frac, width, label="$D_{hard}$", color="#EF4444")
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel("Class fraction")
ax.set_title("Class distribution: hard subset vs full test set")
ax.legend()
for idx, name in enumerate(CLASS_NAMES):
    if RISK_LEVELS[name] == "high":
        ax.get_xticklabels()[idx].set_color("#EF4444")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "hard_subset_class_distribution.png", dpi=150)
plt.show()

dist_table = pd.DataFrame(
    {
        "class": CLASS_NAMES,
        "risk": [RISK_LEVELS[c] for c in CLASS_NAMES],
        "full_frac": full_frac,
        "hard_frac": hard_frac,
        "hard_over_full": np.divide(
            hard_frac, full_frac, out=np.zeros_like(hard_frac),
            where=full_frac > 0,
        ),
    }
)
dist_table.round(4)

## 9. Visualize the 20 hardest cases (highest JS divergence)

For the top-20 most-contested images we show the original dermoscopic image with both agents' predictions, the true label, and the JS divergence. Cells where neither agent matches the ground truth are the prime candidates for the debate machinery to recover the correct diagnosis.

In [ ]:
top_n = min(20, len(hard_df))
top_cases = hard_df.head(top_n).reset_index(drop=True)

if top_n == 0:
    print("No hard cases were flagged; nothing to visualize.")
else:
    cols = 4
    grid_rows = int(np.ceil(top_n / cols))
    fig, axes = plt.subplots(grid_rows, cols, figsize=(4 * cols, 4 * grid_rows))
    axes = np.atleast_1d(axes).ravel()
    for idx in range(len(axes)):
        ax = axes[idx]
        ax.axis("off")
        if idx >= top_n:
            continue
        case = top_cases.iloc[idx]
        try:
            with Image.open(str(case.image_path)) as handle:
                ax.imshow(handle.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE)))
        except Exception as exc:  # noqa: BLE001
            ax.text(0.5, 0.5, f"unreadable\n{exc}", ha="center")
        a_ok = case.pred_a == case.true_label
        b_ok = case.pred_b == case.true_label
        title = (
            f"GT={case.true_label}\n"
            f"A={case.pred_a}{' OK' if a_ok else ' X'} | "
            f"B={case.pred_b}{' OK' if b_ok else ' X'}\n"
            f"JS={case.js_divergence:.3f}"
        )
        colour = "#22C55E" if (a_ok or b_ok) else "#EF4444"
        ax.set_title(title, fontsize=9, color=colour)
    fig.suptitle("20 hardest cases by Jensen-Shannon divergence", y=1.005)
    fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / "hard_subset_top20.png", dpi=150,
                bbox_inches="tight")
    plt.show()

## 10. Summary

The hard subset $D_{hard}$ and its supporting figures are now persisted in `./artifacts`. Notebook **04** consumes the *training-split* analogue of this trigger logic to build consensus features, and notebook **05** re-loads `hard_subset.csv` to report metrics restricted to the contested cases.

In [ ]:
summary = {
    "test_images": int(len(results_df)),
    "hard_cases": int(len(hard_df)),
    "hard_fraction": float(len(hard_df) / max(len(results_df), 1)),
    "mean_js_all": float(results_df["js_divergence"].mean()),
    "mean_js_hard": float(hard_df["js_divergence"].mean()) if len(hard_df) else 0.0,
    "csv_path": str(HARD_SUBSET_CSV.resolve()),
}
for key, value in summary.items():
    print(f"{key:>16}: {value}")